In [54]:
import pandas as pd
import numpy as np

#import csv from 01
df = pd.read_csv("filtered_crmls.csv")
print(df.shape)


(83495, 78)


In [55]:
#drop all rows with missing values
df_clean = df.dropna()
print(df.shape)

(83495, 78)


In [56]:
#identify numeric and categorical columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

numeric_cols, categorical_cols



C:\Users\saman\AppData\Local\Temp\ipykernel_15032\101513429.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()


(['OriginalListPrice',
  'ListingKey',
  'ClosePrice',
  'Latitude',
  'Longitude',
  'LivingArea',
  'ListPrice',
  'DaysOnMarket',
  'FireplacesTotal',
  'AboveGradeFinishedArea',
  'ListingKeyNumeric',
  'TaxAnnualAmount',
  'ParkingTotal',
  'LotSizeAcres',
  'YearBuilt',
  'StreetNumberNumeric',
  'BathroomsTotalInteger',
  'TaxYear',
  'BuildingAreaTotal',
  'BedroomsTotal',
  'ElementarySchoolDistrict',
  'BelowGradeFinishedArea',
  'BusinessType',
  'CoveredSpaces',
  'Stories',
  'LotSizeArea',
  'MainLevelBedrooms',
  'GarageSpaces',
  'AssociationFee',
  'LotSizeSquareFeet',
  'MiddleOrJuniorSchoolDistrict'],
 ['BuyerAgentAOR',
  'ListAgentAOR',
  'Flooring',
  'ViewYN',
  'WaterfrontYN',
  'BasementYN',
  'PoolPrivateYN',
  'ListAgentEmail',
  'CloseDate',
  'ListAgentFirstName',
  'ListAgentLastName',
  'UnparsedAddress',
  'PropertyType',
  'ListOfficeName',
  'BuyerOfficeName',
  'CoListOfficeName',
  'ListAgentFullName',
  'CoListAgentFirstName',
  'CoListAgentLastName'

In [69]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

safe_categorical = [
    "City",
    "CountyOrParish",
    "PostalCode",
    "PropertyType",
    "PropertySubType",
    "StateOrProvince",
    "MLSAreaMajor",
    "ElementarySchoolDistrict",
    "MiddleOrJuniorSchoolDistrict",
    "HighSchoolDistrict"
]

if "CloseDate" not in df.columns:
    df = pd.read_csv("filtered_crmls.csv")

selected_columns = [c for c in safe_categorical + numeric_cols + ["CloseDate"] if c in df.columns]
df = df[selected_columns].copy()
df = df.loc[:, ~df.columns.duplicated()]

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded = encoder.fit_transform(df[safe_categorical])
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(safe_categorical))

In [61]:
df[numeric_cols].isna().sum().sort_values(ascending=False)


TaxAnnualAmount                 83495
AboveGradeFinishedArea          83495
FireplacesTotal                 83495
CoveredSpaces                   83495
BusinessType                    83495
ElementarySchoolDistrict        83495
TaxYear                         83495
MiddleOrJuniorSchoolDistrict    83495
BelowGradeFinishedArea          82892
BuildingAreaTotal               77941
MainLevelBedrooms               32587
AssociationFee                  24041
Stories                          8707
GarageSpaces                     3206
LotSizeAcres                     1442
LotSizeSquareFeet                1442
LotSizeArea                      1436
OriginalListPrice                 176
StreetNumberNumeric               115
YearBuilt                          47
LivingArea                         39
Longitude                          11
Latitude                           11
BathroomsTotalInteger               1
ListingKeyNumeric                   0
ParkingTotal                        0
ListPrice   

In [62]:
#concatenate the encoded categorical columns with the numeric columns

numeric_cols = [ 
    "ClosePrice",
    "DaysOnMarket",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LivingArea",
    "Latitude",
    "Longitude",
    "ListPrice",
    "YearBuilt"
]

df_encoded = pd.concat([
    df[numeric_cols].reset_index(drop=True),
    encoded_df.reset_index(drop=True)
], axis=1)

In [64]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_numeric = scaler.fit_transform(df_encoded[numeric_cols])
df_encoded[numeric_cols] = scaled_numeric



In [71]:
df["CloseDate"] = pd.to_datetime(df["CloseDate"], errors="coerce")
df["Month"] = df["CloseDate"].dt.to_period("M")

latest_month = df["Month"].max()

test_mask = df["Month"] == latest_month

df_test = df_encoded.loc[test_mask].copy()
df_train = df_encoded.loc[~test_mask].copy()

# Export cleaned data for future use
output_path = "cleaned_preprocessed.csv"
df_encoded.to_csv(output_path, index=False)
print(f"Saved cleaned data to {output_path}")
print(df_train.shape)
print(df_test.shape)

Saved cleaned data to cleaned_preprocessed.csv
(71471, 4209)
(12024, 4209)
